In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

import gc

from keras.initializers import Constant
from keras.models import Model
from keras.layers import *

import tensorflow as tf
from keras.callbacks import ModelCheckpoint
from datetime import datetime


tweets = []
labels = []
tests_data = []

def load_tweets(filename, label):
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            tweets.append(line.rstrip())
            labels.append(label)

def load_test_data(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            tests_data.append(line.rstrip())

load_tweets('/content/drive/MyDrive/train_neg_full_cleaned.txt', 0)
load_tweets('/content/drive/MyDrive/train_pos_full_cleaned.txt', 1)
load_test_data('/content/drive/MyDrive/test_cleaned.txt')

# Convert to NumPy array to facilitate indexing
tweets = np.array(tweets)
labels = np.array(labels)
tests = np.array(tests_data)

print(f'{len(tweets)} tweets loaded')



# Load the gloVe embedding
embeddings_index = {}
f = open('/content/drive/MyDrive/glove.twitter.27B.200d.txt')
#f = open('/content/drive/MyDrive/glove_embeddings_20000.txt')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))


max_features = 20000

# Split the data into training and validation sets
train_tweets, val_tweets, train_labels, val_labels = train_test_split(tweets, labels, test_size=0.1, random_state=42)

# Initialize the tokenizer
tokenizer = Tokenizer(num_words=max_features, filters='', split=' ', oov_token="<OOV>")

# Fit the tokenizer on the training tweets
tokenizer.fit_on_texts(train_tweets)

# Convert the training and validation tweets to sequences
train_sequences = tokenizer.texts_to_sequences(train_tweets)
val_sequences = tokenizer.texts_to_sequences(val_tweets)
test_sequences = tokenizer.texts_to_sequences(tests)



# Garbage collect data not used anymore to save memory
del train_tweets, val_tweets, tweets, labels
gc.collect()



word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

num_words = min(max_features, len(word_index)) + 1
print(num_words)

embedding_dim = 200
# First create a matrix of zeros, this is our embedding matrix
embedding_matrix = np.zeros((num_words, embedding_dim))

# For each word in out tokenizer lets try to find that word in the GloVe embedding
for word, i in word_index.items():
    if i > max_features:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # We found the word - add that words vector to the matrix
        embedding_matrix[i] = embedding_vector
    else:
        # Doesn't exist, assign a random vector
        embedding_matrix[i] = np.random.randn(embedding_dim)



# Garbage collect data not used anymore to save memory
del embeddings_index
gc.collect()


# From the above diagram, we see that most sentences are below 40 words
sequence_length = 40
# Pad the sequences
train_padded = pad_sequences(train_sequences, maxlen=sequence_length, padding='post', truncating='post')
val_padded = pad_sequences(val_sequences, maxlen=sequence_length, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=sequence_length, padding='post', truncating='post')


# Garbage collect data not used anymore to save memory
del train_sequences, val_sequences
gc.collect()

del tokenizer
gc.collect()

2500000 tweets loaded
Found 1193514 word vectors.
Found 357231 unique tokens.
20001


0

## Add your model here (as an example, we use MODEL1)

In [ ]:
from keras.initializers import Constant
from keras.models import Model
from keras.layers import *
import tensorflow as tf
from keras import regularizers

conv_filter_size = 128
conv_kernel_size = 2
lstm_unit = 256
num_heads = 8
key_dim = (int)(conv_filter_size / num_heads)
dropout_rate = 0.2

inputs = Input(shape=(sequence_length,), dtype='int32')
embedding_layer = Embedding(num_words, embedding_dim, embeddings_initializer=Constant(embedding_matrix), input_length=sequence_length, trainable=False)(inputs)

# Bidirectional LSTM layer
bilstm_layer = Bidirectional(LSTM(lstm_unit, dropout = 0.2, recurrent_dropout = 0.2, return_sequences=True))(embedding_layer)

# Convolutional layer
conv_layer = Conv1D(filters=conv_filter_size, kernel_size=conv_kernel_size, activation='relu', padding="valid", kernel_initializer="he_uniform")(bilstm_layer)

# Multi-Head Attention layer
multi_head_attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(conv_layer, conv_layer)
multi_head_attention = Dropout(dropout_rate)(multi_head_attention)

# Concatenate Average and Max Pooling
global_pool = concatenate([GlobalAveragePooling1D()(multi_head_attention), GlobalMaxPooling1D()(multi_head_attention)])

# Dense layer
global_pool = Dense(64, activation = "relu")(global_pool)

# Fully connected output layer
output = Dense(1, activation='sigmoid')(global_pool)

In [ ]:
model = Model(inputs=inputs, outputs=output)

# note we're using binary_crossentropy here instead of categorical
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

# Path to load your model parameters
checkpoint_path = '/content/drive/MyDrive/MODEL1/cp-0004.ckpt'
model.load_weights(checkpoint_path)

In [ ]:
# Do the prediction
y_pred = model.predict(test_padded, verbose = 1, batch_size = 256)

if(len(y_pred) != 10000):
   print('Wrong size')
else:
   y_pred[y_pred <= 0.5] = -1
   y_pred[y_pred > 0.5] = 1
   y_pred = y_pred.astype(int)

   df = pd.read_csv('/content/drive/MyDrive/sample_submission.csv')
   df.Prediction = y_pred

   time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
   filename = "submission" + "_" + time + ".csv"
   store_path = '/content/drive/MyDrive/Submissions/' + filename
   df.to_csv(store_path, index = False)
   print("Submission stored")